# Getting started with geoClassy

geoClassy answers one question: **which named area does this GPS point fall in?**
You give it a GeoJSON file of polygons — typically exported from OpenStreetMap —
it builds a spatial index once, and then classifies points: one at a time, or a
million in a batch. It never touches the network.

This notebook walks through the whole API on real data. Run it top to bottom.
Everything is self-contained apart from two small files in `data/`:

- `data/milan.geojson` — Lombardy (region), Milan (municipality) and Milan's nine
  *municipi* (districts): three nested administrative levels, 11 areas.
- `data/italy.geojson` — Italy and San Marino: a multipart boundary (islands)
  with holes (enclaves).

Both are real OpenStreetMap boundaries, lightly simplified — © OpenStreetMap
contributors, ODbL. See `data/README.md` for how they were made.

```
pip install geoClassy pandas
```

pandas is only needed for the DataFrame section.

In [1]:
import geoClassy

geoClassy.__version__

'0.2.1'

## 1. Load a file

`load()` reads a GeoJSON FeatureCollection, keeps every `Polygon` / `MultiPolygon`
feature that has a `name` property, repairs invalid geometry, and builds a spatial
index. The result is an immutable `Areas` object — safe to share between threads,
and you can hold as many as you like at once.

In [2]:
areas = geoClassy.load("data/milan.geojson")
areas

<Areas: 11 areas, on_overlap='smallest'>

In [3]:
len(areas), areas.names

(11,
 ['Lombardia',
  'Milano',
  'Municipio 1',
  'Municipio 2',
  'Municipio 3',
  'Municipio 4',
  'Municipio 5',
  'Municipio 6',
  'Municipio 7',
  'Municipio 8 di Milano',
  'Municipio 9'])

## 2. One point

Arguments are **(latitude, longitude)** — the order people say them in, not the
`[lon, lat]` order GeoJSON stores them in. A point outside every area returns
`None`, not a sentinel string, so it behaves well in pandas.

In [4]:
areas.locate(45.4642, 9.1900)  # Duomo di Milano

'Municipio 1'

In [5]:
print(areas.locate(45.6983, 9.6773))  # Bergamo: in Lombardy, outside Milan
print(areas.locate(46.0037, 8.9511))  # Lugano, Switzerland: outside everything

Lombardia
None


## 3. A whole DataFrame at once

`locate_many` is the fast path: one index query for the whole batch instead of one
per row. Use it instead of `df.apply(...)`. Missing coordinates (`NaN`) come back
as `None`, so a column with gaps still classifies in one call.

In [6]:
import pandas as pd

landmarks = pd.DataFrame({
    "place": ["Duomo", "Castello Sforzesco", "San Siro", "Linate Airport", "Bergamo", "Lugano", "Unknown"],
    "lat":   [45.4642, 45.4705, 45.4781, 45.4494, 45.6983, 46.0037, float("nan")],
    "lon":   [9.1900,  9.1793,  9.1240,  9.2780,  9.6773,  8.9511,  9.1900],
})
landmarks["area"] = areas.locate_many(landmarks.lat, landmarks.lon)
landmarks

,place,lat,lon,area
0,Duomo,45.4642,9.1900,Municipio 1
1,Castello Sforzesco,45.4705,9.1793,Municipio 1
2,San Siro,45.4781,9.1240,Municipio 7
3,Linate Airport,45.4494,9.2780,Lombardia
4,Bergamo,45.6983,9.6773,Lombardia
5,Lugano,46.0037,8.9511,NaN
6,Unknown,NaN,9.1900,NaN


In [7]:
landmarks.area.value_counts(dropna=False)

area
Municipio 1    2
Lombardia      2
NaN            2
Municipio 7    1
Name: count, dtype: int64

## 4. Keep the properties, not just the name

`full=True` returns the feature's whole `properties` dict, so OSM tags such as
`admin_level` or `wikidata` survive the lookup.

In [8]:
areas.locate(45.4642, 9.1900, full=True)

{'name': 'Municipio 1',
 'boundary': 'administrative',
 'admin_level': '10',
 'type': 'boundary',
 'osm_id': 3952986,
 'wikidata': 'Q2494319'}

## 5. When a point is inside several areas

Milan's file has three nested levels, so the Duomo is inside a *municipio*, inside
Milan, inside Lombardy. `overlapping_pairs()` tells you whether your file has this
situation at all — an empty list means no policy can change any answer.

In [9]:
pairs = areas.overlapping_pairs()
len(pairs), pairs[:5]

(19,
 [('Lombardia', 'Municipio 6'),
  ('Lombardia', 'Milano'),
  ('Lombardia', 'Municipio 1'),
  ('Lombardia', 'Municipio 7'),
  ('Lombardia', 'Municipio 8 di Milano')])

By default geoClassy returns the **smallest** matching area — the most specific
one — which is why `locate()` above said `Municipio 1` rather than `Milano`. The
answer does not depend on the order of features in the file. The other policies:

| `on_overlap` | returns |
| --- | --- |
| `"smallest"` | the smallest matching area (default) |
| `"first"` / `"last"` | the first / last match in file order |
| `"all"` | every match, smallest first |
| `"error"` | raises `OverlapError` |

Set one for the whole dataset in `load()`, or override it per call:

In [10]:
point = (45.4642, 9.1900)
for policy in ["smallest", "first", "last", "all"]:
    print(f"{policy:9} -> {areas.locate(*point, on_overlap=policy)}")

smallest  -> Municipio 1
first     -> Lombardia
last      -> Municipio 1
all       -> ['Municipio 1', 'Milano', 'Lombardia']


In [11]:
strict = geoClassy.load("data/milan.geojson", on_overlap="error")
try:
    strict.locate(*point)
except geoClassy.OverlapError as exc:
    print(exc)

point (lat=45.4642, lon=9.19) falls inside 3 areas: Lombardia, Milano, Municipio 1. Pass on_overlap='smallest', 'first', 'last' or 'all'.


If you only want one level, don't lean on a policy — build a dataset that has one
level. `Areas.from_geojson()` takes an already-parsed dict, so you can filter
features in plain Python:

In [12]:
import json

with open("data/milan.geojson", encoding="utf-8") as fh:
    raw = json.load(fh)

districts_only = {
    "type": "FeatureCollection",
    "features": [f for f in raw["features"] if f["properties"]["admin_level"] == "10"],
}
districts = geoClassy.Areas.from_geojson(districts_only)
len(districts), districts.overlapping_pairs()

(9, [])

## 6. Label with a different property

`name` is the default label. Any property works — a code, an English name, a
Wikidata id:

In [13]:
by_wikidata = geoClassy.load("data/milan.geojson", name_key="wikidata")
by_wikidata.locate(45.4642, 9.1900)

'Q2494319'

Ask for a property the file doesn't have and the error lists the ones that exist.
Worth remembering when an export uses `NAME`, `name:en` or `ref` instead of `name`:

In [14]:
try:
    geoClassy.load("data/milan.geojson", name_key="istat_code")
except geoClassy.NoAreasFoundError as exc:
    print(exc)

11 polygonal features found, but none had a 'istat_code' property to use as a label. Available property keys: 'admin_level', 'boundary', 'name', 'osm_id', 'type', 'wikidata'. Pass name_key= to choose one.


## 7. Real boundaries: islands and holes

Administrative boundaries are rarely single rings. Italy is a `MultiPolygon` —
mainland plus islands — and it has **holes** where San Marino and Vatican City
sit. geoClassy handles both without options:

In [15]:
italy = geoClassy.load("data/italy.geojson")
print(italy.names)

checks = {
    "Rome (Colosseum)":         (41.8902, 12.4922),
    "Palermo, Sicily":          (38.1157, 13.3615),
    "Cagliari, Sardinia":       (39.2238,  9.1217),
    "City of San Marino":       (43.9424, 12.4578),
    "St Peter's, Vatican City": (41.9022, 12.4539),
    "Nice, France":             (43.7102,  7.2620),
}
for label, (lat, lon) in checks.items():
    print(f"{label:26} -> {italy.locate(lat, lon)}")

['Italia', 'San Marino']
Rome (Colosseum)           -> Italia
Palermo, Sicily            -> Italia
Cagliari, Sardinia         -> Italia
City of San Marino         -> San Marino
St Peter's, Vatican City   -> None
Nice, France               -> None


San Marino lies inside Italy's outline but resolves to *San Marino* only, and
St Peter's — inside Italy's Vatican hole, with no Vatican polygon in the file —
resolves to nothing. Holes are respected, and the enclave and its host are not
reported as overlapping:

In [16]:
italy.overlapping_pairs()

[]

## 8. Mistakes geoClassy catches — and one it can't

**Swapped coordinates** are the classic error. When the swap pushes a value out of
range you get an explicit message. When both values happen to be valid latitudes —
most of Europe, the eastern United States — it can't tell, and the lookup just
misses:

In [17]:
print(areas.locate(9.1900, 45.4642))  # Milan with lat/lon swapped: 9.19 is a valid latitude, so no error — just None

try:
    areas.locate(139.6917, 35.6895)   # Tokyo with lat/lon swapped: 139 is not a latitude
except ValueError as exc:
    print(exc)

None
latitude 139.692 is out of range [-90, 90]. Did you swap latitude and longitude? geoClassy takes (lat, lon), while GeoJSON coordinates are stored as [lon, lat].


So when *everything* comes back `None`, check one known point by hand before
anything else.

**A point exactly on a boundary counts as inside.** Take a vertex straight from the
data — it sits on the border between two districts, and both claim it:

In [18]:
municipio_1 = next(f for f in raw["features"] if f["properties"]["name"] == "Municipio 1")
geometry = municipio_1["geometry"]
outer_ring = geometry["coordinates"][0][0] if geometry["type"] == "MultiPolygon" else geometry["coordinates"][0]
lon, lat = outer_ring[0]  # GeoJSON order: [lon, lat]
areas.locate(lat, lon, on_overlap="all")

['Municipio 1', 'Municipio 8 di Milano', 'Milano', 'Lombardia']

## 9. Speed

`locate_many` does one index query for the whole batch, so the per-call overhead
disappears. What remains is the geometry test itself, and that scales with how
detailed the polygons are: these are full-resolution OSM boundaries with 400 to
2,000 vertices each, so expect roughly 10 µs per point here, against about 1 µs
on simplified boundaries. If throughput matters, simplifying the polygons (see
`data/README.md`) buys more than anything else. A million random points over the
Milan area:

In [19]:
import time
import numpy as np

rng = np.random.default_rng(0)
lats = rng.uniform(45.38, 45.54, 1_000_000)
lons = rng.uniform(9.04, 9.28, 1_000_000)

t0 = time.perf_counter()
result = areas.locate_many(lats, lons)
print(f"{len(result):,} points in {time.perf_counter() - t0:.2f} s")
pd.Series(result).value_counts(dropna=False)

1,000,000 points in 10.62 s


Lombardia                455195
Municipio 7               93658
Municipio 5               90120
Municipio 8 di Milano     71879
Municipio 9               63141
Municipio 4               61949
Municipio 6               54582
Municipio 3               43359
Municipio 2               38001
Municipio 1               28116
Name: count, dtype: int64

In [20]:
t0 = time.perf_counter()
_ = [areas.locate(lat, lon) for lat, lon in zip(lats[:5000], lons[:5000])]
per_point = (time.perf_counter() - t0) / 5000
print(f"one locate() at a time: {per_point * 1e6:.0f} µs per point — the batch call above is the way to do many")

one locate() at a time: 27 µs per point — the batch call above is the way to do many


## 10. Compressed files

`.geojson.gz` loads transparently. Worth it for country-sized files.

In [21]:
import gzip
import pathlib
import shutil
import tempfile

gz = pathlib.Path(tempfile.mkdtemp()) / "milan.geojson.gz"
with open("data/milan.geojson", "rb") as src, gzip.open(gz, "wb") as dst:
    shutil.copyfileobj(src, dst)

print(f"{gz.stat().st_size / 1024:.0f} KB compressed, "
      f"{pathlib.Path('data/milan.geojson').stat().st_size / 1024:.0f} KB plain")
geoClassy.load(gz).locate(45.4642, 9.1900)

61 KB compressed, 195 KB plain


'Municipio 1'

## 11. Migrating from 0.1.x

The old function API still works, with a `DeprecationWarning`, and will be removed
in 1.0. It deliberately keeps 0.1.x behaviour — the `'unknown'` sentinel, the
`properties.type == "boundary"` filter, and the order-dependent overlap resolution —
so upgrading changes no results until you move to the new API.

| 0.1.x | now |
| --- | --- |
| `single.loadFile(path)` | `areas = geoClassy.load(path)` |
| `single.getNames(lat, lon)` | `areas.locate(lat, lon)` — returns `None`, not `'unknown'` |
| `single.numPoly()` | `len(areas)` |
| `single.polyList()` | `areas.names` |
| `single.checkPoly()` | gone: `load()` validates and repairs |
| `single.requisites()` | gone: just `import geoClassy` |

In [22]:
import warnings

from geoClassy import single

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    single.loadFile("data/milan.geojson")
    print(single.getNames(45.4642, 9.1900), "|", single.getNames(46.0037, 8.9511))

print(caught[0].message)

Municipio 1 | unknown
geoClassy.single.loadFile() is deprecated and will be removed in geoClassy 1.0; use geoClassy.load() instead.


## Where next

- **Getting your own boundaries** — [`docs/getting-data.md`](../docs/getting-data.md):
  Nominatim for one boundary, Overpass for all the subdivisions of an area, why
  `admin_level` means different things in different countries, and why most OSM
  "neighbourhoods" are points rather than polygons.
- **Every option** — the README's *Loading* section.
- **Overlap policy, in depth** — the README's *Overlapping areas* section.